### DPA-DP: extended privacy budget allocation

This work addresses the two main limitations of the first DPA-DP budget allocation test, and re-evaluates the phase-aware privacy budget allocation (thesis' primary contribution) at a scale that produces a usable result.

**Flaws of the first test.** The AIDER dataset's small size required a low noise multiplier, resulting in the final epsilon at unusable 122. Combined with only 50 rounds, the model did not have enough time to learn a disaster signal, and disaster-class accuracy stayed near 1%. Neither result was informative about whether DPA-DP actually helps.

**Changes in this run.** The same MobileNetV3 architecture and AIDER dataset were used, but scaled up: the dataset was multiplied ten times by duplicating images, which permitted higher noise multipliers and brought the final budget down to an acceptable epsilon 7.63. The simulated scenario was extended from 50 to 100 days - three months of normal operation followed by 10 days of disaster - with training twice a day, giving 200 rounds in total. The model configuration was changed by adding one trainable layer (unfrozen classifier plus one backbone layer), matching the model setup used in the surge-detector experiment for consistency.

**Phase switch.** As in the surge-detector experiment, the scenario is split into two segments along the time: an extended normal-operation phase followed by a short simulated disaster phase, with the phase transition triggered at a fixed round. This mirrors the surge detector's design, allowing the two contribution to be merged into one implementation.

**Possible further improvements.**
- Replace image duplication with the already implemented and tested augmentation pipeline, to avoid inflating the dataset with exact copies.
- Introduce a non-IID setting by keeping half the federated clients on the unaffected normal distribution, so the disaster only reaches a part of the federation.
- Further extend dataset size and round count toward a more realistic deployment scale.

**Justification for scaling further.** A real world estimate: New Zealand has approximately 10,000 publicly owned CCTV cameras. If each makes at least one snapshot every 5 minutes and participates in federated training at least once per hour, that gives roughly 12,000 training images per client, about 27 times the size of AIDER. Applied to the same 100-day scenario, this would scale the round count to roughly 2,400.


##### page break

In [ ]:
import pandas
from matplotlib import pyplot
def flwr_accuracy(file, start=0, key="accuracy"):
    df = pandas.read_csv(file + ".csv")
    pyplot.figure(figsize=(8,4))
    for id_val, g in df.groupby("id", sort=False):
        g = g.sort_values("round").tail(len(g) - start)
        pyplot.plot(g["round"], g[key], label=str(id_val))

    pyplot.xlabel("round")
    pyplot.ylabel(key)
    pyplot.grid(True, alpha=0.3)
    pyplot.legend()
    pyplot.tight_layout()
    pyplot.show()

In [ ]:
from matplotlib import pyplot
import opacus.accountants

delta = 1e-5

def epsilon_curve(noise_multiplier_1, noise_multiplier_2, accountant):
    n_batches = (n_samples + batch_size - 1) // batch_size
    sample_rate = 1 / n_batches
    epsilons = [0]
    for round_idx in range(rounds):
        noise_multiplier = noise_multiplier_1 if round_idx < noise_multiplier_switch else noise_multiplier_2
        for _ in range(n_batches):
            accountant.step(noise_multiplier=noise_multiplier, sample_rate=sample_rate)
        epsilons.append(float(accountant.get_epsilon(delta=delta)))
    return epsilons

def plot_epsilon(data):
    pyplot.figure(figsize=(12,4))
    for name, arr in data.items():
        pyplot.plot(arr, label=name)
    pyplot.xlabel("round")
    pyplot.ylabel("epsilon")
    pyplot.grid(True, alpha=0.3)
    pyplot.legend()
    pyplot.show()

##### page break

### DPA-DP privacy budget allocation
Epsilon budget consumption over 200 rounds (180 normal + 20 disaster), for four noise-multiplier strategies tuned to reach a matched final budget of epsilon around 7.63.
- fixed - uniform noise multiplier 1.0 across both phases
- dpa-dp-0.85 - low profile: 1.03 normal, 0.85 disaster
- dpa-dp-0.75 - medium profile: 1.09 normal, 0.75 disaster
- dpa-dp-0.65 - high profile: 1.53 normal, 0.65 disaster, the largest phase difference


In [ ]:
rounds=200
noise_multiplier_switch = 180
n_samples = 4387
batch_size = 32

epsilon_data = {}
epsilon_data["fixed"] = epsilon_curve(1.00, 1.00, opacus.accountants.PRVAccountant())
epsilon_data["dpa-dp-0.85"] = epsilon_curve(1.03, 0.85, opacus.accountants.PRVAccountant())
epsilon_data["dpa-dp-0.75"] = epsilon_curve(1.09, 0.75, opacus.accountants.PRVAccountant())
epsilon_data["dpa-dp-0.65"] = epsilon_curve(1.53, 0.65, opacus.accountants.PRVAccountant())
print({k: a[-1] for k, a in epsilon_data.items()})
plot_epsilon(epsilon_data)

##### page break

### Accuracy between epsilon consumption strategies
tune_layers: 1, learning_rate: 0.01, batch_size: 32, max_grad_norm: 2

As expected, during normal rounds all DPA-DP strategies show slightly worse but comparable accuracy to the fixed baseline. After disaster onset at round 180, the low and medium strategies finish with better accuracy than the fixed baseline. The high-profile strategy 0.65 is a notable outlier, performing worse in both phases.

In [ ]:
flwr_accuracy("budget2", 160)

##### page break

### Disaster accuracy between strategies
Accuracy computed separately for the disaster labels - "fire", "flooded areas", and "collapsed building", since they matter most and require separate attention. Accuracy is zero for all strategies up to round 180, as the model hasn't yet seen disaster images. After round 180, the low and medium profiles show distinguishably better accuracy than the baseline, while the high profile performs worse.


In [ ]:
flwr_accuracy("budget2_sub", 160)

##### page break

### Loss between strategies
Loss is comparable (roughly 12-15) across all strategies except high 0.65, which is around 35. This indicates the 1.53 noise multiplier was high enough to prevent the model from learning traffic incident images, causing it to collapse to an always-predict-normal output during the normal phase.

In [ ]:
flwr_accuracy("budget2", 160, "loss")